# GraphRAG — локальный запуск на Windows
Адаптация `graphtheory-hw2 (4).ipynb`. Исходный ноутбук не изменён.

Окружение: Python 3.12, uv, GraphRAG 3.1.0; установка описана в README_LOCAL.md.
Ollama запускается отдельно. Нужны `qwen3.5-graph:latest` и `bge-m3:latest`.
Запускайте ячейки сверху вниз. По умолчанию обрабатывается первый документ полностью, отчёты по сообществам отключены.
MinerU здесь не запускается: вход — уже распознанные Markdown-файлы.


In [1]:
from pathlib import Path
import sys, os, re, json, hashlib, subprocess, shutil
from importlib.metadata import version
import yaml
import pandas as pd

ROOT = Path.cwd().resolve()
assert (ROOT / "pyproject.toml").exists(), "Запустите Jupyter из папки graph_hw"
assert sys.version_info[:2] == (3, 12), "Выберите ядро из .venv (Python 3.12)"
os.environ["PYTHONUTF8"] = "1"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
print("Python:", sys.version.split()[0], "GraphRAG:", version("graphrag"))
print("Интерпретатор:", sys.executable)

CHAT_MODEL = "qwen3.5-graph:latest"
EMBED_MODEL = "bge-m3:latest"
OLLAMA_URL = "http://127.0.0.1:11434"
DOCUMENTS = ["vanadiy_review.md"]  # Добавьте "ganoshenko.md" для второго графа
SMOKE_TEST = True  # True — только первые SAMPLE_CHARS символов
SAMPLE_CHARS = 6000
GENERATE_COMMUNITY_REPORTS = False  # True — вернуть отчёты и их эмбеддинги

Python: 3.12.13 GraphRAG: 3.1.0
Интерпретатор: c:\Users\qa1ro\OneDrive\Рабочий стол\projects\graph_hw\.venv\Scripts\python.exe


## Проверка Ollama и моделей
Ячейка выполняет по одному короткому запросу к моделям через тот же адаптер, что использует GraphRAG. Скачивание моделей выполняется командами из README.

In [2]:
import urllib.request
from litellm import completion, embedding

try:
    with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=10) as response:
        installed = {item["name"] for item in json.load(response)["models"]}
except Exception as exc:
    raise RuntimeError("Ollama недоступна. Запустите приложение Ollama или ollama serve.") from exc
missing = {CHAT_MODEL, EMBED_MODEL} - installed
if missing:
    raise RuntimeError(f"Не найдены модели: {sorted(missing)}. См. README_LOCAL.md")

reply = completion(
    model=f"ollama_chat/{CHAT_MODEL}", api_base=OLLAMA_URL,
    messages=[{"role": "user", "content": "Ответь одним словом: готов"}],
    reasoning_effort="none", max_tokens=64, timeout=180,
)
assert reply.choices[0].message.content, "Модель вернула пустой ответ"
vectors = embedding(model=f"openai/{EMBED_MODEL}", api_base=f"{OLLAMA_URL}/v1",
                    api_key="ollama", input=["Микролегирование стали"], timeout=180)
EMBED_DIM = len(vectors.data[0]["embedding"])
assert EMBED_DIM == 1024, f"Неожиданная размерность bge-m3: {EMBED_DIM}"
print("Ответ:", reply.choices[0].message.content, "Размерность:", EMBED_DIM)

21:18:17 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
21:18:17 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


Ответ: Готов Размерность: 1024


## Подготовка входа и настроек
Для каждого документа создаётся отдельный граф. Base64 изображений удаляется только
из рабочей копии; подписи сохраняются. Информация внутри картинок автоматически не распознаётся.
Папка запуска зависит от текста, модели и настроек: пробный и полный графы не смешиваются.
Повторный запуск сохраняет кэш. Для независимого эксперимента измените RUN_TAG.

Новые инструкции ограничивают извлечение металлургическими понятиями и требуют цитат. Изменение инструкций создаёт новую папку запуска; старые графы сохраняются. Нормализация после индексации создаёт отдельное представление для просмотра; исходный индекс GraphRAG и его эмбеддинги не меняются.
`GENERATE_COMMUNITY_REPORTS = False` пропускает генерацию отчётов и их эмбеддингов. Выделение сообществ, граф и эмбеддинги сущностей/текста сохраняются. Глобальный поиск по отчётам в этом режиме недоступен. Переключение режима создаёт отдельную папку результатов.

In [3]:
RUN_TAG = "metallurgy-v2"
from graph_quality import TYPES, VERSION
from graphrag.prompts.index.community_report import COMMUNITY_REPORT_PROMPT
prompt_texts = {name: (ROOT / "prompts" / name).read_text(encoding="utf-8") for name in ["extract_metallurgy.txt", "summarize_metallurgy.txt"]}
prompt_texts["community_metallurgy.txt"] = COMMUNITY_REPORT_PROMPT + "\nПиши по-русски. Используй только данные отчёта. Не добавляй внешние знания. Сохрани условия и отрицания. Противоречия обозначай явно. Соблюдай исходную JSON-схему и ссылки на записи.\n"
from graphrag.config.load_config import load_config

jobs = []
for name in DOCUMENTS:
    source = ROOT / name
    raw = source.read_text(encoding="utf-8")
    cleaned, removed = re.subn(
        r"data:image/[^;,\s]+;base64,[A-Za-z0-9+/=\r\n]+",
        "image-omitted", raw,
    )
    text = cleaned[:SAMPLE_CHARS] if SMOKE_TEST else cleaned
    assert text.strip(), f"Пустой документ: {source}"
    settings = {
        "concurrent_requests": 1,
        "completion_models": {"default_completion_model": {
            "type": "litellm", "model_provider": "ollama_chat",
            "model": CHAT_MODEL, "api_base": OLLAMA_URL, "api_key": "ollama",
            "call_args": {"temperature": 0.1, "max_tokens": 3072,
                          "timeout": 600, "reasoning_effort": "none"},
        }},
        "embedding_models": {"default_embedding_model": {
            "type": "litellm", "model_provider": "openai",
            "model": EMBED_MODEL, "api_base": f"{OLLAMA_URL}/v1", "api_key": "ollama",
            "call_args": {"timeout": 180},
        }},
        "input": {"type": "text", "file_pattern": r".*\.md\Z", "encoding": "utf-8"},
        "chunking": {"size": 600, "overlap": 120},
        "extract_graph": {
            "completion_model_id": "default_completion_model",
            "entity_types": TYPES,
            "max_gleanings": 0,
        },
        "summarize_descriptions": {"completion_model_id": "default_completion_model",
                                   "max_length": 500, "max_input_length": 4000},
        "community_reports": {"completion_model_id": "default_completion_model",
                              "max_length": 1500, "max_input_length": 4000},
        "embed_text": {"embedding_model_id": "default_embedding_model", "batch_size": 1},
        "snapshots": {"graphml": True},
    }
    if not GENERATE_COMMUNITY_REPORTS:
        settings["workflows"] = [
            "load_input_documents", "create_base_text_units", "create_final_documents",
            "extract_graph", "finalize_graph", "extract_covariates",
            "create_communities", "create_final_text_units", "generate_text_embeddings",
        ]
        settings["embed_text"]["names"] = ["entity_description", "text_unit_text"]
    print("Отчёты по сообществам:", "включены" if GENERATE_COMMUNITY_REPORTS else "отключены")
    signature = hashlib.sha256((text + json.dumps(settings, sort_keys=True)
                                + json.dumps(prompt_texts, sort_keys=True)
                                + (ROOT / "graph_quality.py").read_text(encoding="utf-8")
                                + RUN_TAG + version("graphrag")).encode()).hexdigest()[:12]
    mode = "sample" if SMOKE_TEST else "full"
    project = ROOT / "local_runs" / f"{source.stem}-{mode}-{signature}"
    (project / "input").mkdir(parents=True, exist_ok=True)
    (project / "input" / source.name).write_text(text, encoding="utf-8")
    prompt_dir = project / "prompts"
    prompt_dir.mkdir(exist_ok=True)
    for prompt_name, content in prompt_texts.items():
        (prompt_dir / prompt_name).write_text(content, encoding="utf-8")
    settings["extract_graph"]["prompt"] = str(prompt_dir / "extract_metallurgy.txt")
    settings["summarize_descriptions"]["prompt"] = str(prompt_dir / "summarize_metallurgy.txt")
    settings["community_reports"]["graph_prompt"] = str(prompt_dir / "community_metallurgy.txt")
    settings.update({
        "input_storage": {"type": "file", "base_dir": str(project / "input")},
        "output_storage": {"type": "file", "base_dir": str(project / "output" / "artifacts")},
        "update_output_storage": {"type": "file", "base_dir": str(project / "update_output")},
        "cache": {"type": "json", "storage": {"type": "file", "base_dir": str(project / "cache")}},
        "reporting": {"type": "file", "base_dir": str(project / "output" / "reports")},
        "vector_store": {"type": "lancedb", "db_uri": str(project / "output" / "lancedb"),
                         "vector_size": 1024},
    })
    (project / "settings.yaml").write_text(yaml.safe_dump(settings, allow_unicode=True), encoding="utf-8")
    config = load_config(project)
    assert config.concurrent_requests == 1
    assert config.vector_store.vector_size == 1024
    (project / "run_info.json").write_text(json.dumps({
        "community_reports_enabled": GENERATE_COMMUNITY_REPORTS, "pipeline_version": VERSION, "source": name, "source_sha256": hashlib.sha256(raw.encode()).hexdigest(),
        "sample": SMOKE_TEST, "characters": len(text), "images_removed": removed,
        "graphrag": version("graphrag"), "chat_model": CHAT_MODEL,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    jobs.append(project)
    print(f"{name}: {len(raw):,} → {len(cleaned):,} символов, удалено изображений: {removed}")
    print("Вход:", len(text), "символов; папка:", project)

Отчёты по сообществам: отключены
vanadiy_review.md: 26,657 → 26,657 символов, удалено изображений: 0
Вход: 6000 символов; папка: C:\Users\qa1ro\OneDrive\Рабочий стол\projects\graph_hw\local_runs\vanadiy_review-sample-6b216c568da9


## Построение графа
Это длительная ячейка. Ждите завершения процесса. Прерывание останавливает только запущенный ею процесс. Кэш не удаляется. При первом запуске библиотека токенизации может скачать свой словарь.

In [4]:
for project in jobs:
    print("Обработка:", project.name, flush=True)
    log_path = project / "index.log"
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(
            [sys.executable, "-m", "graphrag", "index", "--root", str(project)],
            cwd=project, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace",
            env={**os.environ, "PYTHONUTF8": "1", "PYTHONUNBUFFERED": "1"},
        )
        try:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = proc.wait()
        except KeyboardInterrupt:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            raise
    if return_code != 0:
        raise RuntimeError(f"GraphRAG завершился с кодом {return_code}. Лог: {log_path}")
    print("Готово:", project)

Обработка: vanadiy_review-sample-6b216c568da9
21:18:38 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
21:18:38 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
Starting pipeline with workflows: load_input_documents, create_base_text_units, create_final_documents, extract_graph, finalize_graph, extract_covariates, create_communities, create_final_text_units, generate_text_embeddings
Starting workflow: load_input_documents

Workflow complete: load_input_documents
Starting workflow: create_base_text_units
  1 / 1 ............................................................................................

Workflow complete: create_base_text_units
Starting workflow: create_final_documents

Workflow c

## Проверка и упаковка результатов
Архив содержит данные графа, настройки, информацию о запуске и журнал. Для сдачи используйте полный запуск (`SMOKE_TEST = False`), а не пробный. Визуализация — отдельный следующий этап.
После расчёта создаются output/normalized, аудит объединений и manual_review.csv для проверки 20 связей. Наличие точной цитаты проверяется автоматически, её смысл — человеком. Просмотр: graph-viewer.ipynb.

In [5]:
from zipfile import ZipFile, ZIP_DEFLATED
from graph_quality import normalize_graph
for project in jobs:
    artifacts = project / "output" / "artifacts"
    entities = pd.read_parquet(artifacts / "entities.parquet")
    relations = pd.read_parquet(artifacts / "relationships.parquet")
    assert len(entities) > 0 and len(relations) > 0, "Граф пуст: проверьте лог и ответы модели"
    print(project.name, "Узлов:", len(entities), "Связей:", len(relations))
    normalized_dir, quality = normalize_graph(artifacts)
    print("Контроль качества:", quality)
    print("Для ручной проверки:", normalized_dir / "manual_review.csv")
    display(entities.head(), relations.head())
    archive = project.with_suffix(".zip")
    with ZipFile(archive, "w", ZIP_DEFLATED) as z:
        for path in (project / "output").rglob("*"):
            if path.is_file():
                z.write(path, path.relative_to(project))
        for path in (project / "prompts").glob("*.txt"):
            z.write(path, path.relative_to(project))
        for name in ["settings.yaml", "run_info.json", "index.log"]:
            z.write(project / name, name)
    print("Архив:", archive)

vanadiy_review-sample-6b216c568da9 Узлов: 41 Связей: 55
Контроль качества: {'version': 'metallurgy-v2', 'raw_nodes': 41, 'nodes': 41, 'raw_edges': 55, 'edges': 55, 'isolated_nodes': 9, 'nodes_to_review': 0, 'edges_without_quotes': 1}
Для ручной проверки: C:\Users\qa1ro\OneDrive\Рабочий стол\projects\graph_hw\local_runs\vanadiy_review-sample-6b216c568da9\output\normalized\manual_review.csv


,id,human_readable_id,title,type,description,text_unit_ids,frequency,degree
0,2f58ff96-88f7-42ce-8e93-4b4f9a6ded35,0,ВАНАДИЙ,ХИМИЧЕСКИЙ_ЭЛЕМЕНТ,"Ванадий является микролегирующим элементом, ни...",[43fc10d4e2a5f34faf746af8f39d99b060fc041b85b8b...,3,12
1,1928bdd6-b423-4cc5-949d-aeca3bc56a02,1,НИОБИЙ,ХИМИЧЕСКИЙ_ЭЛЕМЕНТ,"Ниобий — это микролегирующий элемент, располож...",[43fc10d4e2a5f34faf746af8f39d99b060fc041b85b8b...,4,21
2,25e5c58d-0998-4ab4-b7a5-86c110a13f1b,2,СТАЛЬ,МАТЕРИАЛ,"СТАЛЬ — это конструкционный материал, подверга...",[43fc10d4e2a5f34faf746af8f39d99b060fc041b85b8b...,3,3
3,a4e3871c-3304-46dd-ab7a-1e1e74cc2f16,3,АУСТЕНИТ,МИКРОСТРУКТУРА,"Аустенит — это фазовая структура стали, подвер...",[43fc10d4e2a5f34faf746af8f39d99b060fc041b85b8b...,5,6
4,14f0790a-b45e-46a5-aa29-cd202405cfe8,4,НИТРИД ВАНАДИЯ,СОЕДИНЕНИЕ,Сущность «НИТРИД ВАНАДИЯ» описывается как соед...,[43fc10d4e2a5f34faf746af8f39d99b060fc041b85b8b...,2,5


,id,human_readable_id,source,target,description,weight,combined_degree,text_unit_ids
0,067aeb16-72d3-4c23-bc95-71ed0587c771,0,НИТРИД ВАНАДИЯ,АУСТЕНИТ,Нитрид ванадия полностью растворяется в аустен...,2.0,11,[43fc10d4e2a5f34faf746af8f39d99b060fc041b85b8b...
1,a5c7d791-4b27-4d18-bb85-dde5b04a456d,1,КАРБИД ВАНАДИЯ,АУСТЕНИТ,Карбид ванадия полностью растворяется в аустен...,2.0,10,[43fc10d4e2a5f34faf746af8f39d99b060fc041b85b8b...
2,ad0ad2b5-5633-4afc-91c1-f0185b10afc0,2,НИТРИД ВАНАДИЯ,РАЗМЕР ЗЕРНА АУСТЕНИТА,Нитрид ванадия не оказывает практически никако...,1.0,7,[43fc10d4e2a5f34faf746af8f39d99b060fc041b85b8b...
3,1f6e2f5f-8c88-476c-9805-f32f76cc5e99,3,КАРБИД ВАНАДИЯ,РАЗМЕР ЗЕРНА АУСТЕНИТА,Карбид ванадия не оказывает практически никако...,1.0,6,[43fc10d4e2a5f34faf746af8f39d99b060fc041b85b8b...
4,0db03719-33b3-4f99-9b27-6841377f1cf7,4,НИОБИЙ,СТАЛЬ,Ниобий применяется для микролегирования констр...,15.0,24,[43fc10d4e2a5f34faf746af8f39d99b060fc041b85b8b...


Архив: C:\Users\qa1ro\OneDrive\Рабочий стол\projects\graph_hw\local_runs\vanadiy_review-sample-6b216c568da9.zip
